In [1]:
# Hybrid Retriver - Combining Dense and spares retriver

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document


e:\RAG LEARNING\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

In [3]:
type(docs)

list

In [3]:
# step-2: Dense Retriver (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs,embedding_model)
dense_retriver = dense_vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 777.06it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Sparse Regtirver(BM25)
sparse_retriver = BM25Retriever.from_documents(docs)
sparse_retriver.k = 3 # top k document to retriver


In [8]:
from langchain_classic.retrievers import EnsembleRetriever

# step4 : combine with ensemble Retriver 
hybrid_retriver = EnsembleRetriever(
    retrievers = [dense_retriver,sparse_retriver],
    weight = [0.7,0.3]
)

In [9]:
hybrid_retriver

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001749B23C440>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001749B25ED50>, k=3)], weights=[0.5, 0.5])

In [10]:
# Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriver.invoke(query)

# step 6 : print results
for i,doc in enumerate(results):
    print(f"\nDocument {i+1}:\n{doc.page_content}")
    


Document 1:
LangChain helps build LLM applications.

Document 2:
Langchain can be used to develop agentic ai application.

Document 3:
Langchain has many types of retrievers.

Document 4:
Pinecone is a vector database for semantic search.


In [12]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

In [13]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [17]:
# Step 5: Prompt Template
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")


llm=init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq"
    )
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001749E511640>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000174A02081A0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [14]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [18]:
# Create stuff Document Chain
document_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)

# Create full RAG chain
rag_chain = create_retrieval_chain(retriever=hybrid_retriver,combine_docs_chain=document_chain)

In [20]:
# step 9 : Ask a question
query={"input":"How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

# step 10 : Output
print("Answer:\n",response["answer"])
print("\n Source Documents")
for i,doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

Answer:
 Based on the context, you can build an app using LLMs (Large Language Models) with LangChain. LangChain is a tool that helps developers build LLM applications. It provides a framework to integrate LLMs with other technologies to create more complex AI applications.

To build an app using LLMs with LangChain, you can follow these general steps:

1. Choose a LangChain retriever: LangChain supports various types of retrievers, which are responsible for fetching relevant data from a database or other sources. You can select a retriever that suits your needs, such as a database retriever or a text retriever.
2. Integrate with a vector database (optional): If you want to implement semantic search, you can use a vector database like Pinecone. This will allow your app to search for information based on the semantic meaning of the text, rather than just exact matches.
3. Develop your LLM application: With the retriever and vector database (if used) set up, you can now use LangChain to 